# InsightFuel Analytics — 03. Gold

Transformação da camada **Silver** em tabelas analíticas da camada **Gold**.

**Entradas**
- `silver.combustivel_tratado`
- `silver.municipio_tratado`
- `silver.indicadores_municipio_tratado`

**Saídas**
- `gold.fato_preco_mensal`
- `gold.visao_municipio`
- `gold.visao_regional`


## 1. Configuração e conexão

In [1]:
from pathlib import Path
import re
import unicodedata
import pandas as pd
import duckdb

def find_project_root(start=Path.cwd()):
    start = Path(start).resolve()
    for p in [start] + list(start.parents):
        if (p / "data" / "bronze").exists():
            return p
    raise FileNotFoundError("Pasta data/bronze não encontrada.")

PROJECT_ROOT = find_project_root()
DB_PATH = PROJECT_ROOT / "database" / "insightfuel.duckdb"

if not DB_PATH.exists():
    raise FileNotFoundError(
        "DuckDB não encontrado. Execute primeiro os notebooks 01_ingestao e 02_silver."
    )

con = duckdb.connect(str(DB_PATH))
print("Projeto:", PROJECT_ROOT)
print("DuckDB:", DB_PATH)


Projeto: C:\Users\Bruna\anaconda_projects\Trabalho_Armazenamento
DuckDB: C:\Users\Bruna\anaconda_projects\Trabalho_Armazenamento\database\insightfuel.duckdb


## 2. Verificar as tabelas Silver

In [2]:
silver_tables = con.execute("""
    SELECT table_name
    FROM information_schema.tables
    WHERE table_schema = 'silver'
    ORDER BY table_name
""").df()

display(silver_tables)

required = {
    "combustivel_tratado",
    "municipio_tratado",
    "indicadores_municipio_tratado"
}
missing = required - set(silver_tables["table_name"])
if missing:
    raise ValueError(f"Tabelas Silver ausentes: {missing}")

print("✓ Todas as tabelas Silver estão disponíveis.")


,table_name
0,combustivel_tratado
1,indicadores_municipio_tratado
2,municipio_tratado


✓ Todas as tabelas Silver estão disponíveis.


## 3. Inspecionar as colunas Silver

In [3]:
for table in sorted(required):
    print(f"\n===== silver.{table} =====")
    display(con.execute(f"DESCRIBE silver.{table}").df())



===== silver.combustivel_tratado =====


,column_name,column_type,null,key,default,extra
0,regiao_sigla,VARCHAR,YES,None,None,None
1,estado_sigla,VARCHAR,YES,None,None,None
2,municipio,VARCHAR,YES,None,None,None
3,revenda,VARCHAR,YES,None,None,None
4,cnpj_da_revenda,VARCHAR,YES,None,None,None
5,nome_da_rua,VARCHAR,YES,None,None,None
6,numero_rua,VARCHAR,YES,None,None,None
7,complemento,VARCHAR,YES,None,None,None
8,bairro,VARCHAR,YES,None,None,None
9,cep,VARCHAR,YES,None,None,None



===== silver.indicadores_municipio_tratado =====


,column_name,column_type,null,key,default,extra
0,ano,BIGINT,YES,None,None,None
1,codigo_da_grande_regiao,BIGINT,YES,None,None,None
2,nome_da_grande_regiao,VARCHAR,YES,None,None,None
3,codigo_da_unidade_da_federacao,BIGINT,YES,None,None,None
4,sigla_da_unidade_da_federacao,VARCHAR,YES,None,None,None
5,nome_da_unidade_da_federacao,VARCHAR,YES,None,None,None
6,codigo_do_municipio,BIGINT,YES,None,None,None
7,nome_do_municipio,VARCHAR,YES,None,None,None
8,regiao_metropolitana,VARCHAR,YES,None,None,None
9,codigo_da_mesorregiao,BIGINT,YES,None,None,None



===== silver.municipio_tratado =====


,column_name,column_type,null,key,default,extra
0,uf,VARCHAR,YES,None,None,None
1,cod_uf,DOUBLE,YES,None,None,None
2,cod_munic,DOUBLE,YES,None,None,None
3,nome_do_municipio,VARCHAR,YES,None,None,None
4,populacao_apurada_ibge_censo_demografico_2022_...,VARCHAR,YES,None,None,None
5,codigo_ibge,VARCHAR,YES,None,None,None


## 4. Funções auxiliares

In [4]:
def normalize_name(value):
    value = "" if pd.isna(value) else str(value).strip().lower()
    value = unicodedata.normalize("NFKD", value).encode("ascii", "ignore").decode("ascii")
    return re.sub(r"[^a-z0-9]+", "_", value).strip("_")

def find_column(columns, aliases):
    normalized = {normalize_name(c): c for c in columns}
    for alias in aliases:
        alias = normalize_name(alias)
        if alias in normalized:
            return normalized[alias]
    return None

def require_column(df, aliases, description):
    col = find_column(df.columns, aliases)
    if col is None:
        raise ValueError(
            f"Não foi possível identificar '{description}'.\n"
            f"Colunas disponíveis:\n{list(df.columns)}"
        )
    print(f"{description}: {col}")
    return col

def to_numeric_br(series):
    if pd.api.types.is_numeric_dtype(series):
        return pd.to_numeric(series, errors="coerce")
    cleaned = (
        series.astype("string")
        .str.strip()
        .str.replace(".", "", regex=False)
        .str.replace(",", ".", regex=False)
    )
    return pd.to_numeric(cleaned, errors="coerce")

def normalize_ibge_code(series):
    """Normaliza códigos IBGE para texto de 7 dígitos, sem perder zeros à esquerda."""
    s = series.astype("string").str.strip()
    # Remove o sufixo .0 comum quando o Excel foi lido como número.
    s = s.str.replace(r"\.0$", "", regex=True)
    s = s.str.extract(r"(\d+)", expand=False)
    return s.str.zfill(7)

def make_city_key(uf_series, municipio_series):
    """Cria chave estável UF|MUNICIPIO, removendo acentos e diferenças de caixa/pontuação."""
    uf = uf_series.map(normalize_name).str.upper()
    municipio = municipio_series.map(normalize_name).str.upper()
    return uf + "|" + municipio


## 5. Carregar dados da Silver

In [5]:
anp = con.execute("SELECT * FROM silver.combustivel_tratado").df()
pop = con.execute("SELECT * FROM silver.municipio_tratado").df()
pib = con.execute("SELECT * FROM silver.indicadores_municipio_tratado").df()

print("ANP:", anp.shape)
print("População:", pop.shape)
print("PIB:", pib.shape)


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

ANP: (2619039, 19)
População: (5602, 6)
PIB: (77965, 44)


## 6. Identificar os campos da ANP

A `fato_preco_mensal` terá granularidade **mês × UF × município × produto**.


In [7]:
date_start = require_column(
    anp,
    ["data_inicial", "data_inicio", "data_da_coleta", "data_coleta", "data"],
    "data inicial/coleta"
)

uf_col = require_column(
    anp, ["estado", "uf", "sigla_uf", "estado_sigla"], "UF"
)

municipio_col = require_column(
    anp, ["municipio", "nome_municipio", "nome_do_municipio"], "município"
)

produto_col = require_column(
    anp, ["produto", "nome_produto"], "produto"
)

price_col = require_column(
    anp,
    [
        "preco_medio_revenda",
        "preco_medio_revend",
        "preco_revenda",
        "preco_medio",
        "valor_de_venda",
        "valor_venda",
        "preco"
    ],
    "preço"
)


data inicial/coleta: data_da_coleta
UF: estado_sigla
município: municipio
produto: produto
preço: valor_de_venda


## 7. Preparar ANP para a granularidade mensal

In [8]:
anp_gold = anp.copy()

anp_gold["data_referencia"] = pd.to_datetime(
    anp_gold[date_start], errors="coerce", dayfirst=True
)
anp_gold["mes"] = anp_gold["data_referencia"].dt.to_period("M").dt.to_timestamp()
anp_gold["uf"] = anp_gold[uf_col].astype("string").str.strip().str.upper()
anp_gold["municipio"] = anp_gold[municipio_col].astype("string").str.strip()
anp_gold["produto"] = anp_gold[produto_col].astype("string").str.strip()
anp_gold["preco"] = to_numeric_br(anp_gold[price_col])

anp_gold = anp_gold.dropna(
    subset=["mes", "uf", "municipio", "produto", "preco"]
)

print("Registros válidos:", f"{len(anp_gold):,}")
display(anp_gold[["mes", "uf", "municipio", "produto", "preco"]].head())


Registros válidos: 2,619,039


,mes,uf,municipio,produto,preco
0,2021-01-01,RS,SAO LEOPOLDO,GASOLINA,4.599
1,2021-01-01,RS,SAO LEOPOLDO,ETANOL,4.199
2,2021-01-01,RS,SAO LEOPOLDO,GASOLINA ADITIVADA,4.799
3,2021-01-01,RS,SAO LEOPOLDO,DIESEL,3.499
4,2021-01-01,RS,SAO LEOPOLDO,DIESEL S10,3.599


## 8. Construir `gold.fato_preco_mensal`

In [9]:
fato_preco_mensal = (
    anp_gold
    .groupby(["mes", "uf", "municipio", "produto"], as_index=False)
    .agg(
        preco_medio=("preco", "mean"),
        preco_minimo=("preco", "min"),
        preco_maximo=("preco", "max"),
        desvio_padrao=("preco", "std"),
        quantidade_observacoes=("preco", "count")
    )
)

fato_preco_mensal["desvio_padrao"] = fato_preco_mensal["desvio_padrao"].fillna(0)
fato_preco_mensal["ano"] = fato_preco_mensal["mes"].dt.year
fato_preco_mensal["mes_numero"] = fato_preco_mensal["mes"].dt.month

display(fato_preco_mensal.head())
print("Dimensão:", fato_preco_mensal.shape)


,mes,uf,municipio,produto,preco_medio,preco_minimo,preco_maximo,desvio_padrao,quantidade_observacoes,ano,mes_numero
0,2021-01-01,AC,CRUZEIRO DO SUL,DIESEL,5.240000,5.23,5.25,0.010000,7,2021,1
1,2021-01-01,AC,CRUZEIRO DO SUL,DIESEL S10,5.260000,5.23,5.30,0.029439,10,2021,1
2,2021-01-01,AC,CRUZEIRO DO SUL,GASOLINA,5.939000,5.92,5.95,0.011972,10,2021,1
3,2021-01-01,AC,CRUZEIRO DO SUL,GASOLINA ADITIVADA,5.935000,5.93,5.95,0.010000,4,2021,1
4,2021-01-01,AC,RIO BRANCO,DIESEL,4.548093,4.30,4.80,0.104752,86,2021,1


Dimensão: (72412, 11)


## 9. Preparar dimensão municipal

In [10]:
pop_gold = pop.copy()

pop_code = find_column(
    pop_gold.columns,
    ["codigo_ibge", "codigo_municipio", "cod_munic", "cod_municipio", "cod_ibge"]
)
pop_uf_code = find_column(pop_gold.columns, ["cod_uf", "codigo_uf"])
pop_nome = find_column(
    pop_gold.columns,
    ["nome_do_municipio", "municipio", "nome_municipio", "nome"]
)
pop_uf = find_column(pop_gold.columns, ["uf", "sigla_uf", "estado_sigla"])

print("Código município:", pop_code)
print("Código UF:", pop_uf_code)
print("Nome município:", pop_nome)
print("UF:", pop_uf)

# Monta o código IBGE da forma mais confiável disponível.
if pop_code:
    pop_gold["codigo_ibge"] = normalize_ibge_code(pop_gold[pop_code])
    # Se o arquivo traz apenas os 5 dígitos do município, combina com o código da UF.
    if pop_uf_code:
        raw_code = pop_gold[pop_code].astype("string").str.replace(r"\.0$", "", regex=True)
        only_digits = raw_code.str.extract(r"(\d+)", expand=False)
        uf_digits = pop_gold[pop_uf_code].astype("string").str.extract(r"(\d+)", expand=False)
        mask_5 = only_digits.str.len().eq(5) & uf_digits.notna()
        pop_gold.loc[mask_5, "codigo_ibge"] = (
            uf_digits[mask_5].str.zfill(2) + only_digits[mask_5].str.zfill(5)
        )
elif pop_uf_code and pop_nome:
    pop_gold["codigo_ibge"] = pd.NA
else:
    pop_gold["codigo_ibge"] = pd.NA

if pop_nome:
    pop_gold["municipio"] = pop_gold[pop_nome].astype("string").str.strip()
else:
    pop_gold["municipio"] = pd.NA

if pop_uf:
    pop_gold["uf"] = pop_gold[pop_uf].astype("string").str.strip().str.upper()
elif pop_uf_code:
    # Não há necessidade de converter o código da UF em sigla aqui; a chave por nome
    # será usada quando disponível.
    pop_gold["uf"] = pd.NA
else:
    pop_gold["uf"] = pd.NA

pop_gold["chave_municipio"] = make_city_key(
    pop_gold["uf"], pop_gold["municipio"]
)

display(
    pop_gold[
        ["codigo_ibge", "uf", "municipio", "chave_municipio"]
    ].head()
)
print("Códigos IBGE válidos:", pop_gold["codigo_ibge"].notna().sum())
print("Chaves município válidas:", pop_gold["chave_municipio"].notna().sum())


Código município: codigo_ibge
Código UF: cod_uf
Nome município: nome_do_municipio
UF: uf


,codigo_ibge,uf,municipio,chave_municipio
0,0000015,RO,Alta Floresta D'Oeste,RO|ALTA_FLORESTA_D_OESTE
1,0000023,RO,Ariquemes,RO|ARIQUEMES
2,0000031,RO,Cabixi,RO|CABIXI
3,0000049,RO,Cacoal,RO|CACOAL
4,0000056,RO,Cerejeiras,RO|CEREJEIRAS


Códigos IBGE válidos: 5570
Chaves município válidas: 5602


## 10. Identificar população

In [11]:
population_col = find_column(
    pop_gold.columns,
    [
        "populacao_apurada_ibge_no_censo_demografico_2022",
        "populacao",
        "populacao_2022",
        "populacao_censo_2022"
    ]
)

if population_col is None:
    candidates = [c for c in pop_gold.columns if "populacao" in normalize_name(c)]
    if len(candidates) == 1:
        population_col = candidates[0]

print("Coluna de população:", population_col)

if population_col is None:
    raise ValueError(
        "Não foi possível identificar a população. "
        f"Colunas disponíveis: {list(pop_gold.columns)}"
    )

pop_gold["populacao"] = to_numeric_br(pop_gold[population_col])

# Mantém tanto o código IBGE quanto a chave UF|município.
municipio_dim = (
    pop_gold[
        ["codigo_ibge", "uf", "municipio", "chave_municipio", "populacao"]
    ]
    .drop_duplicates(subset=["chave_municipio"], keep="last")
    .copy()
)

display(municipio_dim.head())
print("Municípios na dimensão:", len(municipio_dim))


Coluna de população: populacao_apurada_ibge_censo_demografico_2022_e_malha_territorial_2023


,codigo_ibge,uf,municipio,chave_municipio,populacao
0,0000015,RO,Alta Floresta D'Oeste,RO|ALTA_FLORESTA_D_OESTE,21494
1,0000023,RO,Ariquemes,RO|ARIQUEMES,96833
2,0000031,RO,Cabixi,RO|CABIXI,5351
3,0000049,RO,Cacoal,RO|CACOAL,86887
4,0000056,RO,Cerejeiras,RO|CEREJEIRAS,15890


Municípios na dimensão: 5602


## 11. Preparar PIB municipal

In [17]:
pib_gold = pib.copy()

pib_code = find_column(
    pib_gold.columns,
    ["codigo_ibge", "codigo_municipio", "cod_municipio", "cod_munic", "cod_ibge"]
)
pib_year = find_column(
    pib_gold.columns,
    ["ano", "ano_referencia", "ano_de_referencia", "periodo"]
)
pib_total = find_column(
    pib_gold.columns,
    [
        "pib",
        "produto_interno_bruto",
        "produto_interno_bruto_a_precos_correntes",
        "produto_interno_bruto_a_precos_correntes_r_1_000"
    ]
)
pib_per_capita = find_column(
    pib_gold.columns,
    [
        "pib_per_capita",
        "pib_per_capita_r",
        "pib_per_capita_reais",
        "produto_interno_bruto_per_capita",
        "produto_interno_bruto_per_capita_r",
        "produto_interno_bruto_per_capita_a_precos_correntes_r_1_00"
    ]
)
pib_nome = find_column(
    pib_gold.columns,
    ["municipio", "nome_municipio", "nome_do_municipio", "nome"]
)
pib_uf = find_column(
    pib_gold.columns,
    ["uf", "sigla_uf", "estado_sigla", "sigla_da_uf", "sigla_da_unidade_da_federacao"]
)

print("Código:", pib_code)
print("Ano:", pib_year)
print("PIB:", pib_total)
print("PIB per capita:", pib_per_capita)
print("Município:", pib_nome)
print("UF:", pib_uf)

if pib_year is None:
    raise ValueError(
        "Não foi possível identificar o ano do PIB. "
        f"Colunas disponíveis: {list(pib_gold.columns)}"
    )

if pib_code is None and (pib_nome is None or pib_uf is None):
    raise ValueError(
        "Não foi possível identificar uma chave de município no PIB. "
        "É necessário código IBGE ou município + UF."
    )

# Código IBGE, quando disponível.
if pib_code:
    pib_gold["codigo_ibge"] = normalize_ibge_code(pib_gold[pib_code])
else:
    pib_gold["codigo_ibge"] = pd.NA

# Ano.
pib_gold["ano"] = pd.to_numeric(
    pib_gold[pib_year].astype("string").str.extract(r"(\d{4})", expand=False),
    errors="coerce"
)

# Indicadores.
pib_gold["pib"] = (
    to_numeric_br(pib_gold[pib_total])
    if pib_total else pd.NA
)
pib_gold["pib_per_capita"] = (
    to_numeric_br(pib_gold[pib_per_capita])
    if pib_per_capita else pd.NA
)

# Chave alternativa UF|município. Ela permite recuperar o PIB mesmo quando
# o código IBGE da população não coincide com o código vindo do arquivo do PIB.
if pib_nome and pib_uf:
    pib_gold["municipio"] = pib_gold[pib_nome].astype("string").str.strip()
    pib_gold["uf"] = pib_gold[pib_uf].astype("string").str.strip().str.upper()
    pib_gold["chave_municipio"] = make_city_key(
        pib_gold["uf"], pib_gold["municipio"]
    )
else:
    pib_gold["chave_municipio"] = pd.NA

pib_dim = (
    pib_gold[
        [
            "codigo_ibge",
            "ano",
            "chave_municipio",
            "pib",
            "pib_per_capita"
        ]
    ]
    .dropna(subset=["ano"])
    .copy()
)

# Uma linha por chave de município + ano.
# Primeiro removemos duplicidades exatas; depois mantemos uma linha por chave.
pib_dim = pib_dim.drop_duplicates(
    subset=["codigo_ibge", "ano", "chave_municipio"],
    keep="last"
)

print("Linhas município/ano no PIB:", len(pib_dim))
print("Anos do PIB:", sorted(pib_dim["ano"].dropna().unique().tolist()))
print("Códigos IBGE no PIB:", pib_dim["codigo_ibge"].notna().sum())
print("Chaves município no PIB:", pib_dim["chave_municipio"].notna().sum())

display(pib_dim.head())


Código: codigo_ibge
Ano: ano
PIB: produto_interno_bruto_a_precos_correntes_r_1_000
PIB per capita: produto_interno_bruto_per_capita_a_precos_correntes_r_1_00
Município: nome_do_municipio
UF: sigla_da_unidade_da_federacao
Linhas município/ano no PIB: 77965
Anos do PIB: [2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023]
Códigos IBGE no PIB: 77965
Chaves município no PIB: 77965


,codigo_ibge,ano,chave_municipio,pib,pib_per_capita
0,1100015,2010,RO|ALTA_FLORESTA_D_OESTE,262076.878,10731.18
1,1100023,2010,RO|ARIQUEMES,1364693.780,15103.86
2,1100031,2010,RO|CABIXI,69611.114,11033.62
3,1100049,2010,RO|CACOAL,1186494.091,15095.15
4,1100056,2010,RO|CEREJEIRAS,222021.189,13037.06


## 12. Construir `gold.visao_municipio`

In [18]:
visao_municipio = fato_preco_mensal.copy()

# Chave canônica da ANP: UF + município, removendo diferenças de acentuação,
# caixa e pontuação.
visao_municipio["chave_municipio"] = make_city_key(
    visao_municipio["uf"],
    visao_municipio["municipio"]
)

# 1) Enriquecimento com população/código IBGE.
municipio_dim_join = municipio_dim.copy()
municipio_dim_join = municipio_dim_join.drop_duplicates(
    subset=["chave_municipio"], keep="last"
)

visao_municipio = visao_municipio.merge(
    municipio_dim_join[
        ["chave_municipio", "codigo_ibge", "populacao"]
    ],
    on="chave_municipio",
    how="left",
    validate="many_to_one"
)

visao_municipio["ano"] = visao_municipio["mes"].dt.year

# 2) PIB: primeiro tenta código IBGE + ano.
pib_por_codigo = (
    pib_dim.dropna(subset=["codigo_ibge"])
    .drop_duplicates(subset=["codigo_ibge", "ano"], keep="last")
    [
        ["codigo_ibge", "ano", "pib", "pib_per_capita"]
    ]
    .rename(columns={
        "pib": "pib_codigo",
        "pib_per_capita": "pib_per_capita_codigo"
    })
)

visao_municipio = visao_municipio.merge(
    pib_por_codigo,
    on=["codigo_ibge", "ano"],
    how="left",
    validate="many_to_one"
)

# 3) Para os casos em que o código não casou, tenta UF + município + ano.
pib_por_chave = (
    pib_dim.dropna(subset=["chave_municipio"])
    .drop_duplicates(subset=["chave_municipio", "ano"], keep="last")
    [
        ["chave_municipio", "ano", "pib", "pib_per_capita"]
    ]
    .rename(columns={
        "pib": "pib_chave",
        "pib_per_capita": "pib_per_capita_chave"
    })
)

visao_municipio = visao_municipio.merge(
    pib_por_chave,
    on=["chave_municipio", "ano"],
    how="left",
    validate="many_to_one"
)

# Usa o resultado por código quando disponível e, caso contrário,
# recupera o PIB pela chave UF|município.
visao_municipio["pib"] = (
    visao_municipio["pib_codigo"]
    .combine_first(visao_municipio["pib_chave"])
)

visao_municipio["pib_per_capita"] = (
    visao_municipio["pib_per_capita_codigo"]
    .combine_first(visao_municipio["pib_per_capita_chave"])
)

visao_municipio = visao_municipio.drop(
    columns=[
        "pib_codigo",
        "pib_chave",
        "pib_per_capita_codigo",
        "pib_per_capita_chave"
    ]
)

for col in ["populacao", "pib", "pib_per_capita"]:
    visao_municipio[col] = pd.to_numeric(
        visao_municipio[col], errors="coerce"
    )

display(
    visao_municipio[
        [
            "mes", "ano", "uf", "municipio",
            "codigo_ibge", "preco_medio",
            "populacao", "pib", "pib_per_capita"
        ]
    ].head(20)
)

print("Dimensão:", visao_municipio.shape)
print(
    "Cobertura PIB:",
    f"{visao_municipio['pib'].notna().mean():.1%}"
)
print(
    "Cobertura PIB per capita:",
    f"{visao_municipio['pib_per_capita'].notna().mean():.1%}"
)


,mes,ano,uf,municipio,codigo_ibge,preco_medio,populacao,pib,pib_per_capita
0,2021-01-01,2021,AC,CRUZEIRO DO SUL,0000203,5.240000,91888,2.058630e+06,22934.82
1,2021-01-01,2021,AC,CRUZEIRO DO SUL,0000203,5.260000,91888,2.058630e+06,22934.82
2,2021-01-01,2021,AC,CRUZEIRO DO SUL,0000203,5.939000,91888,2.058630e+06,22934.82
3,2021-01-01,2021,AC,CRUZEIRO DO SUL,0000203,5.935000,91888,2.058630e+06,22934.82
4,2021-01-01,2021,AC,RIO BRANCO,0000401,4.548093,364756,1.095567e+07,26119.02
5,2021-01-01,2021,AC,RIO BRANCO,0000401,4.600147,364756,1.095567e+07,26119.02
6,2021-01-01,2021,AC,RIO BRANCO,0000401,4.051259,364756,1.095567e+07,26119.02
7,2021-01-01,2021,AC,RIO BRANCO,0000401,5.103633,364756,1.095567e+07,26119.02
8,2021-01-01,2021,AC,RIO BRANCO,0000401,5.175698,364756,1.095567e+07,26119.02
9,2021-01-01,2021,AL,ARAPIRACA,0000300,3.868258,234696,5.916238e+06,25249.73


Dimensão: (72412, 16)
Cobertura PIB: 99.8%
Cobertura PIB per capita: 99.8%


## 13. Construir `gold.visao_regional`

In [19]:
visao_regional = (
    fato_preco_mensal
    .groupby(["mes", "ano", "mes_numero", "uf", "produto"], as_index=False)
    .agg(
        preco_medio=("preco_medio", "mean"),
        preco_minimo=("preco_minimo", "min"),
        preco_maximo=("preco_maximo", "max"),
        desvio_padrao=("preco_medio", "std"),
        quantidade_municipios=("municipio", "nunique"),
        quantidade_observacoes=("quantidade_observacoes", "sum")
    )
)

visao_regional["desvio_padrao"] = visao_regional["desvio_padrao"].fillna(0)
display(visao_regional.head())


,mes,ano,mes_numero,uf,produto,preco_medio,preco_minimo,preco_maximo,desvio_padrao,quantidade_municipios,quantidade_observacoes
0,2021-01-01,2021,1,AC,DIESEL,4.894047,4.300,5.25,0.489252,2,93
1,2021-01-01,2021,1,AC,DIESEL S10,4.930074,4.360,5.30,0.466586,2,105
2,2021-01-01,2021,1,AC,ETANOL,4.051259,3.800,4.28,0.000000,1,58
3,2021-01-01,2021,1,AC,GASOLINA,5.521316,4.919,5.95,0.590694,2,108
4,2021-01-01,2021,1,AC,GASOLINA ADITIVADA,5.555349,5.030,5.95,0.536908,2,90


## 14. Gravar as tabelas Gold

In [20]:
con.execute("CREATE SCHEMA IF NOT EXISTS gold")

con.register("tmp_fato", fato_preco_mensal)
con.execute("""
    CREATE OR REPLACE TABLE gold.fato_preco_mensal AS
    SELECT * FROM tmp_fato
""")

con.register("tmp_municipio", visao_municipio)
con.execute("""
    CREATE OR REPLACE TABLE gold.visao_municipio AS
    SELECT * FROM tmp_municipio
""")

con.register("tmp_regional", visao_regional)
con.execute("""
    CREATE OR REPLACE TABLE gold.visao_regional AS
    SELECT * FROM tmp_regional
""")

print("✓ Tabelas Gold gravadas com sucesso.")


✓ Tabelas Gold gravadas com sucesso.


## 15. Testes de qualidade

In [21]:
gold_tables = con.execute("""
    SELECT table_name
    FROM information_schema.tables
    WHERE table_schema = 'gold'
    ORDER BY table_name
""").df()

display(gold_tables)

expected = {
    "fato_preco_mensal",
    "visao_municipio",
    "visao_regional"
}
missing = expected - set(gold_tables["table_name"])
assert not missing, f"Tabelas Gold ausentes: {missing}"

counts = con.execute("""
    SELECT
        (SELECT COUNT(*) FROM gold.fato_preco_mensal) AS fato_preco_mensal,
        (SELECT COUNT(*) FROM gold.visao_municipio) AS visao_municipio,
        (SELECT COUNT(*) FROM gold.visao_regional) AS visao_regional
""").df()

display(counts)
assert (counts.iloc[0] > 0).all()

negative_prices = con.execute("""
    SELECT COUNT(*)
    FROM gold.fato_preco_mensal
    WHERE preco_medio < 0
""").fetchone()[0]
assert negative_prices == 0, "Há preços médios negativos."

duplicates = con.execute("""
    SELECT mes, uf, municipio, produto, COUNT(*) AS qtd
    FROM gold.fato_preco_mensal
    GROUP BY ALL
    HAVING COUNT(*) > 1
""").df()
assert duplicates.empty, "Há duplicidades na granularidade da fato."

print("✓ Testes de qualidade concluídos.")


,table_name
0,fato_preco_mensal
1,visao_municipio
2,visao_regional


,fato_preco_mensal,visao_municipio,visao_regional
0,72412,72412,5494


✓ Testes de qualidade concluídos.


## 16. Cobertura dos enriquecimentos

In [22]:
coverage = con.execute("""
    SELECT
        COUNT(*) AS total_linhas,
        COUNT(codigo_ibge) AS linhas_com_codigo_ibge,
        COUNT(populacao) AS linhas_com_populacao,
        COUNT(pib) AS linhas_com_pib,
        COUNT(pib_per_capita) AS linhas_com_pib_per_capita
    FROM gold.visao_municipio
""").df()

coverage["pct_codigo_ibge"] = coverage["linhas_com_codigo_ibge"] / coverage["total_linhas"] * 100
coverage["pct_populacao"] = coverage["linhas_com_populacao"] / coverage["total_linhas"] * 100
coverage["pct_pib"] = coverage["linhas_com_pib"] / coverage["total_linhas"] * 100
coverage["pct_pib_per_capita"] = coverage["linhas_com_pib_per_capita"] / coverage["total_linhas"] * 100

display(coverage)


,total_linhas,linhas_com_codigo_ibge,linhas_com_populacao,linhas_com_pib,linhas_com_pib_per_capita,pct_codigo_ibge,pct_populacao,pct_pib,pct_pib_per_capita
0,72412,72277,71710,72277,72277,99.813567,99.030547,99.813567,99.813567


# 17. Resultado da camada Gold

```text
Silver
   │
   ├── combustivel_tratado
   ├── municipio_tratado
   └── indicadores_municipio_tratado
           │
           ▼
        Gold
           ├── fato_preco_mensal
           ├── visao_municipio
           └── visao_regional
                  │
                  ▼
          análises e gráficos
```

A Gold é a camada destinada às análises exploratórias e visualizações.

Indicadores mais específicos, como paridade etanol/gasolina, preço esperado versus observado, índice de valor do combustível e detecção de outliers, podem ser adicionados sobre essas tabelas.


## 18. Encerrar conexão

In [23]:
con.close()
print("Conexão com o DuckDB encerrada.")


Conexão com o DuckDB encerrada.
